# LLM-JEPA Symbolic Regression: Inference & Evaluation

Use this notebook to load trained models from Google Drive and run symbolic regression inference.

## 1. Setup & Mount Drive

In [ ]:
import os, re
!git clone https://github.com/udohchuks/GSOC-LM-JEPA_for_Symbolic_Regression.git
%cd GSOC-LM-JEPA_for_Symbolic_Regression
!pip install -r requirements.txt

from google.colab import drive
drive.mount('/content/drive')

## 2. Locate Best Checkpoint

Sorts all checkpoints by their embedded validation loss and picks the lowest.

In [ ]:
drive_ckpt_dir = '/content/drive/MyDrive/SymbolicRegression/checkpoints'

# Parse the loss from the filename and sort ascending; best = [-1] after reversing, or [0].
ckpts = sorted(
    [f for f in os.listdir(drive_ckpt_dir) if f.endswith('.ckpt') and 'val' in f],
    key=lambda f: float(re.search(r'total=([0-9.]+)', f).group(1))
)

# [0] = lowest loss = best. Fallback to last.ckpt if no val-named files exist.
best_ckpt_name = ckpts[0] if ckpts else 'last.ckpt'
ckpt_path = os.path.join(drive_ckpt_dir, best_ckpt_name)
print(f'Best checkpoint: {best_ckpt_name}')

## 3. Run Inference on Equation

In [ ]:
EQ_ID = 'I.6.2a'  # change to any AI Feynman equation ID
!python predict.py --ckpt {ckpt_path} --eq_id {EQ_ID}

## 4. Full Evaluation Suite

In [ ]:
!python run_eval.py --ckpt {ckpt_path} --output ./results/drive_eval_results.json

import json
with open('./results/drive_eval_results.json') as f:
    m = json.load(f)
print(f"Exact Recovery Rate: {m['exact_recovery_rate']*100:.1f}%")
print(f"Mean R2 (post-BFGS): {m['mean_r2_post_bfgs']:.4f}")